# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
ds = mlc.Dataset(croissant_url)
metadata = ds.metadata

print(f"Dataset title: {metadata.name}")
print(f"Description: {metadata.description}\n")
print(f"License: {metadata.license if hasattr(metadata, 'license') else getattr(metadata, 'license_', None)}")
print(f"Published: {metadata.datePublished if hasattr(metadata, 'datePublished') else getattr(metadata, 'date_published', None)}")
print(f"Keywords: {', '.join(metadata.keywords)}\n")

## 2. Data Overview
Review available record sets, fields, and their IDs. This helps to structure how data can be loaded and referenced using their `@id` fields.

In [ ]:
from pprint import pprint

# List all record set @ids
record_sets = []
if hasattr(metadata, 'recordSet') and metadata.recordSet:
    if isinstance(metadata.recordSet, list):
        record_sets = [rs['@id'] if isinstance(rs, dict) and '@id' in rs else rs for rs in metadata.recordSet]
    else:
        record_sets = [metadata.recordSet['@id'] if isinstance(metadata.recordSet, dict) and '@id' in metadata.recordSet else metadata.recordSet]
else:
    # If list is empty, attempt listing from the Croissant schema (dataset.metadata.to_json())
    loaded_metadata = ds.metadata.to_json() if hasattr(ds.metadata, 'to_json') else {}
    # Fallback for notebook robustness
    if 'recordSet' in loaded_metadata:
        record_sets = [rs['@id'] if isinstance(rs, dict) and '@id' in rs else rs for rs in loaded_metadata['recordSet']]

print(f"Available record sets (by @id):")
for rs in record_sets:
    print(f" - {rs}")

# Preview record set fields and columns
def show_record_set_info(record_set_id):
    print(f"\nRecord set: {record_set_id}\n{'-'*32}")
    # Using ds.records yields dicts with field @ids as keys
    try:
        # Get a single record to list fields/columns
        for rec in ds.records(record_set=record_set_id):
            pprint(rec)
            break
    except Exception as e:
        print(f"Could not inspect record set {record_set_id} due to: {e}")

for rsid in record_sets:
    show_record_set_info(rsid)

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview. For this example, we'll use the main protein data record set.

In [ ]:
# List of record set @ids (identified above)
record_set_ids = record_sets.copy()
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Extracting records from {record_set_id} ...")
    records = list(ds.records(record_set=record_set_id))
    try:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records.")
        print(f"Columns: {df.columns.tolist()}\n")
    except Exception as e:
        print(f"Could not convert record set {record_set_id} to DataFrame. Error: {e}")

# If there are multiple record sets, pick the most relevant one (usually the largest/main table)
if len(dataframes) > 0:
    main_record_set_id = max(dataframes, key=lambda k: len(dataframes[k]))
    print(f"Selected main record set for analysis: {main_record_set_id}\n")
    print(f"Sample data (first 5 rows):")
    display(dataframes[main_record_set_id].head())
else:
    main_record_set_id = None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes filtering records, transforming numeric fields, or grouping data by key attributes. All field columns are referenced by their `@id` keys.

In [ ]:
if main_record_set_id:
    df = dataframes[main_record_set_id]
    print(f"Columns in main record set (@id): {df.columns.tolist()}")
    # Guess a numeric field. Typical names: coverage_percent, MW, peptide_count, etc. Use first numeric column found.
    numeric_field = None
    for col in df.columns:
        try:
            # Check if conversion is possible
            df[col] = pd.to_numeric(df[col], errors='coerce')
            if df[col].notnull().sum() > 0 and (df[col].dtype == float or df[col].dtype == int):
                numeric_field = col
                break
        except:
            continue

    print(f"\nUsing numeric field for filtering: {numeric_field}")
    if numeric_field is not None:
        threshold = df[numeric_field].mean() # Example threshold: mean value
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold:.2f}")
        print(filtered_df[[numeric_field]].head())

        # Normalization (zero mean, unit std)
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()
        print(f"\nNormalized '{numeric_field}' values (first 5 rows):")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try to group by a likely categorical field (e.g. protein family, accession, or description fields)
        group_field = None
        for possible in ['accession', 'description', 'sample', 'protein_group']:
            if possible in df.columns:
                group_field = possible
                break
        if group_field is not None:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"\nMean {numeric_field} grouped by '{group_field}' (first 5 groups):")
            print(grouped_df.head())
    else:
        print("No numeric field found suitable for analysis.")
else:
    print("No main record set DataFrame available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Below, we show example visualizations for the main numeric field and its normalized distribution by group.

In [ ]:
if main_record_set_id and numeric_field is not None:
    plt.figure(figsize=(7, 4))
    df[numeric_field].hist(bins=30, alpha=0.7, color='tab:blue')
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.title(f'Distribution of {numeric_field}')
    plt.grid(axis='y', alpha=0.2)
    plt.show()

    if group_field is not None:
        plt.figure(figsize=(10, 5))
        grouped_sample = grouped_df.copy()
        # Show only the first 20 groups for clarity if too many
        if len(grouped_sample) > 20:
            grouped_sample = grouped_sample.head(20)
        plt.bar(grouped_sample[group_field], grouped_sample[numeric_field], color='tab:orange')
        plt.xticks(rotation=45, ha='right')
        plt.ylabel(f'Mean {numeric_field}')
        plt.xlabel(group_field)
        plt.title(f'Mean {numeric_field} by {group_field}')
        plt.tight_layout()
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to load a Croissant-format dataset with `mlcroissant`, view record sets and their field `@id`s, and load data for exploratory analysis. Numeric protein features were filtered and normalized, and simple visualizations of their distributions and summary statistics were provided.

This workflow can be adapted to deeper biological analysis, e.g., for comparing protein abundances, studying peptide modifications, or integrating experimental metadata.